# Object Detection with a Lightweight Neural Network (MobileNet-SSD)

## Libraries and settings

In [ ]:
# Libraries
import os
import cv2
import numpy as np
import requests
import matplotlib.pyplot as plt

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Current working directory
print('Current working directory:', os.getcwd())


## Load the pretrained MobileNet-SSD model

MobileNet-SSD is a small ('mobile-friendly') Caffe model pretrained on the PASCAL VOC dataset (20 object classes). It needs two files: the network architecture (`deploy.prototxt`) and the trained weights (`mobilenet_iter_73000.caffemodel`, ~23 MB).

In [ ]:
# Download the pretrained model files (only needs to run once)
files = {
    'deploy.prototxt': 'https://raw.githubusercontent.com/chuanqi305/MobileNet-SSD/master/deploy.prototxt',
    'mobilenet_iter_73000.caffemodel': 'https://raw.githubusercontent.com/chuanqi305/MobileNet-SSD/master/mobilenet_iter_73000.caffemodel',
}

for filename, url in files.items():
    if not os.path.exists(filename):
        r = requests.get(url)
        with open(filename, 'wb') as f:
            f.write(r.content)

# Load the model with cv2.dnn.readNetFromCaffe
net = cv2.dnn.readNetFromCaffe('deploy.prototxt', 'mobilenet_iter_73000.caffemodel')
print('Model loaded', f'({os.path.getsize("mobilenet_iter_73000.caffemodel") / 1e6:.1f} MB)')


## Read your image

Replace `image_url` below with a link to your own `.jpg` (e.g. from Google image search).

In [ ]:
# Download an example image (replace this with your own .jpg from Google image search)
image_url = 'https://raw.githubusercontent.com/AlexeyAB/darknet/master/data/dog.jpg'
image_path = 'my_image.jpg'

if not os.path.exists(image_path):
    r = requests.get(image_url)
    with open(image_path, 'wb') as f:
        f.write(r.content)

# Read the image
image = cv2.imread(image_path)

# Change the color to RGB (from BGR) for display later
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Plot the image
plt.figure(figsize=(6, 6))
plt.imshow(image_rgb)
plt.axis('off')
plt.show()


## VOC class names

MobileNet-SSD was pretrained on the PASCAL VOC dataset, which has 20 object classes.

In [ ]:
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat',
    'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant',
    'sheep', 'sofa', 'train', 'tvmonitor'
]
print(f'{len(VOC_CLASSES)} classes (including background)')


## Object detection function

Preprocess the image into a blob, run a forward pass, and filter the resulting detections by a confidence threshold.

In [ ]:
def detect_objects(net, image, conf_threshold=0.4):
    h, w = image.shape[:2]

    # Create a blob from the image (resize to 300x300, scale by 0.007843 = 1/127.5, subtract mean 127.5)
    blob = cv2.dnn.blobFromImage(cv2.resize(image, (300, 300)), 0.007843, (300, 300), 127.5)

    # Set the blob as input and run a forward pass
    net.setInput(blob)
    output = net.forward()

    # output has shape (1, 1, num_detections, 7); each row is:
    # [image_id, class_id, confidence, x1, y1, x2, y2] with box coords normalized to [0, 1]
    result = image.copy()
    detections = []

    for i in range(output.shape[2]):
        confidence = output[0, 0, i, 2]
        if confidence < conf_threshold:
            continue

        class_id = int(output[0, 0, i, 1])
        label = VOC_CLASSES[class_id]

        box = output[0, 0, i, 3:7] * np.array([w, h, w, h])
        x1, y1, x2, y2 = box.astype(int)

        detections.append((label, round(float(confidence), 3)))
        cv2.rectangle(result, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(result, f'{label} {confidence:.2f}', (x1, max(y1 - 8, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    return result, detections


## Run detection

In [ ]:
result, detections = detect_objects(net, image_rgb, conf_threshold=0.4)

plt.figure(figsize=(8, 8))
plt.imshow(result)
plt.axis('off')
plt.show()

print('Detected objects:', detections)


**Image used:** `dog.jpg` (the classic multi-object demo photo — a dog, a bicycle, and a car).

**Detected objects (confidence threshold 0.4):** `bicycle` (0.997), `car` (0.990), `dog` (0.912) — all three real objects in the photo, correctly labeled.

## Experiment with the confidence threshold

In [ ]:
thresholds = [0.5, 0.97, 0.995]
fig, axes = plt.subplots(1, len(thresholds), figsize=(15, 5))

for ax, t in zip(axes, thresholds):
    res, dets = detect_objects(net, image_rgb, conf_threshold=t)
    ax.imshow(res)
    ax.set_title(f'threshold = {t}\n{len(dets)} detections')
    ax.axis('off')
    print(f'threshold={t}: {len(dets)} detections -> {dets}')

plt.tight_layout()
plt.show()


**Reflection:** All three detections are already highly confident (≥0.91) on this image, and the model doesn't produce any low-confidence false-positive boxes here (the other 97 candidate detections it considers all score exactly 0.0) — so a low threshold changes nothing. Raising the threshold instead progressively trims the *weakest correct* detections: at threshold=0.5 all three objects remain; at 0.97 `dog` (0.912) is dropped, leaving `bicycle` and `car`; at 0.995 `car` (0.990) is dropped too, leaving only `bicycle` (0.997). So here the threshold mainly trades off recall (keeping all real objects) against precision on borderline-confident detections, rather than filtering out noise.

### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')
